In [ ]:
%cd ..
#from sklearnex import patch_sklearn
#patch_sklearn()

from rag import ask, match_cv, code_cv, build_index
import pandas as pd
import numpy as np
import json

from sklearn.preprocessing import MinMaxScaler

In [ ]:
import torch

if torch.cuda.is_available():
    free_memory, total_memory = torch.cuda.mem_get_info()
    print(f"Free Memory: {free_memory / 1e9:.2f} GB")
    print(f"Total Memory: {total_memory / 1e9:.2f} GB")
    print(f"Used Memory: {(total_memory - free_memory) / 1e9:.2f} GB")

In [ ]:
jobs_df = pd.read_parquet("_jobs_backups/jobs_20260526_201606.parquet")

# verify index integrity
if not jobs_df.set_index("key").index.is_unique:
    dupes = jobs_df[jobs_df["key"].duplicated(keep=False)].sort_values("key")
    print(dupes[["key", "title", "employer_name", "city", "found_on"]])

    raise Exception("Manually delete duplicate rows before proceeding")


In [ ]:
from scraper.sources._utils import parse_posted_at

def filter_recent(df, cutoff: str):
    """Keep rows posted on/after `cutoff`, e.g. '7 days ago' or '1 month ago'.
    Rows with missing posted_at are kept."""
    if cutoff.strip().isdigit() or "ago" not in cutoff.lower():
        raise ValueError(
            f"cutoff must be a relative phrase like '7 days ago', got {cutoff!r}"
        )
    cutoff_date = parse_posted_at(cutoff)
    if cutoff_date is None:
        raise ValueError(f"could not parse cutoff: {cutoff!r}")
    return df[df["posted_at"].isna() | (df["posted_at"] >= cutoff_date)]

# usage
recent = filter_recent(jobs_df, "3 weeks ago")
#last_month = filter_recent(df, "1 month ago")
# TODO: wire this below

recent

In [ ]:
# sanity check (perform only if CHROMA_DIR is set or you'll waste compute)

index = build_index(jobs_df)
hits = index.search("data science", k=1500)
df_keys = set(jobs_df["key"].astype(str))
print(f"all hits in df: {all(h['key'] in df_keys for h in hits)}")
print(f"hits: {len(hits)}")

## Ask

In [ ]:
# answer, answer_df = ask("jobs about climate or progressive causes", jobs_df, k=20)
# print(answer)
# answer_df

## Match CV

In [ ]:
# Step 1: thematic coding. Inspect and edit before scoring.
cv_text = open("LM_CV.md").read()

# profile = code_cv(cv_text)
# print(profile["role_themes"])
# print(profile["must_have_themes"])
# print(profile["disqualifiers"])

with open("base_profile.json", "r") as f:
    profile = json.load(f)
    print(json.dumps(profile, indent=4))

### Prescreen only mode (to run on smaller model separately)
Then remember to add the corresponding `previous_run_dir` in config [match.dump] (or it's all pointless)

In [ ]:
# top = match_cv(profile, cv_text, jobs_df, top_n_retrieve=1500, prescreen_only=True)

Have you set the previous_run_dir ?

In [ ]:
# Step 2: score and rank
top = match_cv(profile, cv_text, jobs_df, top_n_retrieve=1500)

In [ ]:
from rag.match_graph import _recruiter_score

weights = {
    "technical_match": 0.40,
    "seniority_match": 0.60,
    "transferability": 0.0,
    "trajectory": 0.0,
    "soft_skills": 0.0,
}


top["recruiter_score"] = top.apply(lambda row: _recruiter_score(row, weights=weights), axis=1)

In [ ]:
# ghosts from a previous keyword based coding
#top.drop(columns=["matched_by", "matched_by_n", "score"], inplace=True)

# rescale 0-1 both scores
scaler = MinMaxScaler(feature_range=(0, 1))
top["recr_rescaled"] = scaler.fit_transform(top[["recruiter_score"]])
top["cand_rescaled"] = scaler.fit_transform(top[["candidate_score"]])

In [ ]:
# define weighted average params
recr_weight = 0.6
cand_weight = 0.4
w_mean = lambda r: np.average(
    [r["recr_rescaled"],r["cand_rescaled"]], 
    weights = [recr_weight,cand_weight]
)

top["combined_score"] = top.apply(w_mean,axis=1)

In [ ]:
# define minimum thresholds
recruiter_thr = 65
#candidate_thr = 0.5 # applied to rescaled one

survivors = top[top["recruiter_score"]>=recruiter_thr]
#survivors = survivors[survivors["cand_rescaled"]>=candidate_thr]

In [ ]:

survivors = survivors.sort_values(["combined_score"], ascending=False).reset_index(drop=True)

survivors

In [ ]:
for (_, row) in survivors.sort_values(["recruiter_score"], ascending = False).iterrows():
    display(row.to_dict())
    print(80*"-"+"\n")
